# 1 — Buffers, layouts, tensors (and charts)

The core objects, bottom-up:

- a **Buffer** is rank-1 featureless storage — bytes, an address space, a length;
- a **Layout** gives coordinates to those bytes via the contract
  `loc = offset + Σ stride_d · i_d` with each raw lattice coordinate in
  `[start_d, stop_d)`;
- a **Tensor** is `Buffer + DType + Layout`;
- a **Chart** (step 3) labels a dim's lattice with exact physical
  coordinates: `phys(i) = origin + i·step`.

Design decisions D1–D13 are recorded in `../DESIGN.md`.

In [1]:
import numpy as np
from nbhelp import formula, show  # also puts tensorlib on sys.path
from tensorlib import Buffer, Dim, Layout, Tensor, q

## A buffer is just bytes

Twelve float32 values → 48 raw bytes. The buffer has a length and a device,
and *no idea* that it holds floats, let alone a matrix.

In [2]:
payload = np.arange(12, dtype=np.float32).tobytes()
buf = Buffer.from_bytes(payload)
buf

Buffer(48B @ cpu)

## A layout gives the bytes coordinates

Dims are built by hand: a name, a **fully expanded byte stride**, and a
half-open lattice domain. Here `i` steps one float (4 bytes) and `j` steps a
whole `i`-column (12 bytes) — so `i` is the fast dimension.

In [3]:
lay = Layout(
    (
        Dim("i", stride=4, start=0, stop=3),  # one float32 apart
        Dim("j", stride=12, start=0, stop=4),  # one whole i-column apart
    )
)
lay

Layout(offset=0, i[0:3)*4, j[0:4)*12)

The contract, evaluated by hand: `loc(i=1, j=2) = 0 + 4·1 + 12·2 = 28`.

In [4]:
lay.get_loc(i=1, j=2)

28

## A tensor = buffer + dtype + layout

In [5]:
t = Tensor(buf, np.float32, lay)
show(t, "hand-built tensor")
formula(t)

-- hand-built tensor
Tensor[float32] on Buffer(48B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i            4      0      3      3  
  j           12      0      4      4  
  numel=12  footprint=(0, 48)  injectivity=injective
loc = 0 + 4*i + 12*j   (bytes)


In [6]:
t.item(i=1, j=2)  # byte 28 / 4 bytes per float = element 7 of the arange

np.float32(7.0)

## Memory order is just strides

Same 48 bytes, second layout with the strides swapped: now `j` is fast.
"Row-major vs column-major" never appears as a concept — it is entirely
carried by the strides.

In [7]:
t.to_numpy()

array([[ 0.,  3.,  6.,  9.],
       [ 1.,  4.,  7., 10.],
       [ 2.,  5.,  8., 11.]], dtype=float32)

In [8]:
lay_jfast = Layout((Dim("i", 16, 0, 3), Dim("j", 4, 0, 4)))
Tensor(buf, np.float32, lay_jfast).to_numpy()

array([[ 0.,  1.,  2.,  3.],
       [ 4.,  5.,  6.,  7.],
       [ 8.,  9., 10., 11.]], dtype=float32)

## Domains need not start at zero

Raw coordinates (D3) mean a negative index is a *literal coordinate*, not
Python's "from the end". The offset is the affine map's constant term: here
it places `(i=-1, j=-2)` at byte 0.

In [9]:
lay_centered = Layout(
    (Dim("i", 4, -1, 2), Dim("j", 12, -2, 2)),
    offset=4 * 1 + 12 * 2,
)
tc = Tensor(buf, np.float32, lay_centered)
show(tc, "centered domains")
tc.item(i=-1, j=-2), tc.item(i=0, j=0)

-- centered domains
Tensor[float32] on Buffer(48B @ cpu)
  offset : 28 bytes
  dim     stride  start   stop   size  chart
  i            4     -1      2      3  
  j           12     -2      2      4  
  numel=12  footprint=(0, 48)  injectivity=injective


(np.float32(0.0), np.float32(7.0))

## Charts: physical labels on the lattice

The raw integer indexing above is *already* a chart — origin `start`,
step 1, dimensionless. Attaching a real chart just widens that to exact
rational origins/steps with units. Nothing about addressing changes.

Here the hand-built scan becomes space × time: `i` is a stage position
(0.25 um pitch), `j` is a sample time (1 ms period).

In [10]:
ct = t.with_charts(i=("0 um", "0.25 um"), j=("0 ms", "1 ms"))
show(ct, "charted: position x time")

-- charted: position x time
Tensor[float32] on Buffer(48B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  i            4      0      3      3  pos[i: 0 um step 0.25 um]
  j           12      0      4      4  pos[j: 0 ms step 1 ms]
  numel=12  footprint=(0, 48)  injectivity=injective


## Both faces of indexing

Lattice ints and physical quantities address the same element — ints always
mean the lattice, quantities go through the chart (exactly).

In [11]:
ct.item(i=1, j=2), ct.item(i=q("0.25 um"), j=q("2 ms")), ct.phys("i", 2)

(np.float32(7.0), np.float32(7.0), 0.5 um)

## Membership is exact-only; snapping is deliberate

An off-lattice coordinate is an *error*, not a rounding. `snap` is the
explicit way to round onto the lattice.

In [12]:
try:
    ct.item(i=q("0.3 um"), j=q("0 ms"))
except ValueError as e:
    print("off-lattice:", e)
ct.snap("i", q("0.3 um"))

off-lattice: 0.3 um is off-lattice for pos[i: 0 um step 0.25 um] (lattice position 6/5); use snap() to round deliberately


1

## Compiler mode

Charts are strictly metadata: strip them and the pure lattice remains,
byte-for-byte identical. Plain-int indexing never needed them in the first
place.

In [13]:
lat = ct.strip_charts()
print(lat.item(i=1, j=2) == ct.item(i=1, j=2))
try:
    lat.item(i=q("0.25 um"), j=0)
except TypeError as e:
    print("no chart:", e)

True
no chart: dim i has no chart; pass a lattice int, not 0.25 um


## Built-in analyses

`footprint` is the byte interval the layout can touch; `injectivity` is a
conservative proof of one-to-one-ness; `check` validates against the buffer.

In [14]:
aliased = Tensor(buf, np.float32, Layout((Dim("x", 4, 0, 3), Dim("r", 0, 0, 5))))
show(aliased, "stride-0 dim: 5 aliases of each element")

-- stride-0 dim: 5 aliases of each element
Tensor[float32] on Buffer(48B @ cpu)
  offset : 0 bytes
  dim     stride  start   stop   size  chart
  x            4      0      3      3  
  r            0      0      5      5  
  numel=15  footprint=(0, 12)  injectivity=aliased


In [15]:
too_big = Layout((Dim("x", 4, 0, 13),))  # 13 floats > 12 in the buffer
try:
    Tensor(buf, np.float32, too_big).check()
except ValueError as e:
    print("rejected:", e)

rejected: layout footprint [0, 52) exceeds buffer [0, 48)
